## This notebook runs on all configurations and genertes the simulated data, It also estimates the power spectrum from the input data which can be plotted.
## all data are generate solely from this notebook. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal.windows import nuttall
from matplotlib.colors import LogNorm
import pkfuncs as pfunc
import scipy
from scipy.signal import windows
import plot_scripts as plts
import os
plt.style.use("mplstyle")

In [2]:
# input model power spectrum
n = 2.0
N = 768
dnuc = 40e-3            # MHz
r = 9209.714526830916 # Mpc
rp = 16.98990953034521  # Mpc/MHz
dL = dnuc * rp
L  = N * dL
Nrea = 100

# EoR 21-cm PS for Mondal+17
PK21 = np.load('PK21.npy')
k_model = PK21[:, 0]
P_model = PK21[:, 1]
# P_model = k_model**(-n)


k_fft = 2*np.pi*np.arange(N)/L

# kper = 0.1
# k_fft = np.sqrt( k_fft**2 + kper**2 ) 

k_abs = np.abs(k_fft)

P_cont = np.interp(k_abs, k_model, P_model , left=0, right=0)

# Discrete variance for FFT modes
P_dft = (N**2 / L) * P_cont 
P_dft = L * P_cont 
# P_dft = P_cont 

# binning 
NB = 64  # number of bins


In [3]:
(N**2 / L), (N / dL),  (N * dL),  (L), dL

(1130.082533147537,
 1130.0825331475373,
 521.9300207722049,
 521.9300207722049,
 0.6795963812138084)

In [4]:
INP_SIGNAL_CONFIGS = {
    "21cm": dict(type="21cm"),
    "FG+21cm": dict(type="FG+21cm"),
    "FG+MM+21cm": dict(type="FG+MM+21cm"),
}


# -----------------------------
# FLAG configurations
# -----------------------------
FLAG_CONFIGS = {
    "NOFLAG": dict(mode="NOFLAG", percent=0, seed=42),
    "RANDOM35": dict(mode="RANDOM", percent=35, seed=42),
    "PERIODIC": dict(mode="PERIODIC", percent=0, seed=42),
    "PERIODIC+RANDOM10": dict(mode="PERIODIC+RANDOM", percent=10, seed=42),
}

# -----------------------------
# SCF configurations
# -----------------------------
SCF_CONFIGS = {
    "None": dict(SCF="None", NN_gp=None, NN_hann=None),
    "GP":   dict(SCF="GP",   NN_gp=96,  NN_hann=None),
    "Hann": dict(SCF="Hann", NN_gp=None, NN_hann=50),
}


In [5]:
def generate_fields(inp_signal, *, N, Nrea, P_dft):
    # 21-cm realizations (always needed)
    fields_21cm = np.array(
        [pfunc.draw_field_from_power(P_dft, seed=55 + i) for i in range(Nrea)]
    )

    if inp_signal == "21cm":
        return fields_21cm

    if inp_signal == "FG+21cm":
        fields_forg = pfunc.generate_gp_realizations(
            Npoints=N,
            amplitude=1e12,
            length_scale=1e4,
            Nreal=Nrea,
            kernel_type="RBF",
            sigma=1,
        )
        return fields_forg + fields_21cm

    if inp_signal == "FG+MM+21cm":
        fields_forg = pfunc.generate_gp_realizations(
            Npoints=N,
            amplitude=1e12,
            length_scale=100,
            Nreal=Nrea,
            kernel_type="RBF",
            sigma=1,
        )
        return fields_forg + fields_21cm

    raise ValueError(f"Unknown inp_signal: {inp_signal}")


In [6]:
def run_one_configuration(
    fields_tota,
    inp_signal,
    flag_cfg,
    scf_cfg,
    *,
    N, L, dL, r, NB,
    k_model, P_model,
    P_cont, k_abs,
    path="results"
):
    # -----------------------------
    os.makedirs(path, exist_ok=True)
    # -----------------------------
    flag = pfunc.flagdata(
        N,
        mode=flag_cfg["mode"],
        percent=flag_cfg["percent"],
        seed=flag_cfg["seed"],
    )

    flag_percent = 100 * (1 - np.count_nonzero(flag) / N)
    print(
        f"[FLAG={flag_cfg['mode']}, SCF={scf_cfg['SCF']}] "
        f"flagged = {flag_percent:.1f}%"
    )

    # -----------------------------
    # SCF processing
    # -----------------------------
    fields_orig, fields_flag, ml = pfunc.process_scf(
        fields_tota,
        flag,
        scf_cfg["SCF"],
        NN_gp=scf_cfg["NN_gp"],
        NN_hann=scf_cfg["NN_hann"],
    )

    # -----------------------------
    # FFT estimator
    # -----------------------------
    pk_fft = pfunc.pk_fft(fields_flag, L, N)

    n1_fft, nend_fft = 0, N // 2
    kb_fft, mean_fft, err_fft, binned_th_fft, _ = pfunc.bin_power_spectrum(
        n1_fft, nend_fft,
        k_abs[n1_fft:nend_fft],
        pk_fft,
        P_cont,
        NB
    )

    # -----------------------------
    # DCT estimator
    # -----------------------------
    M = np.argmax(ml.real == 0) if np.any(ml.real == 0) else ml.shape[0]
    k_dct = np.pi * np.arange(M) / ((M - 1) * dL)
    # print(k_dct)
    
    P_cont_dct = np.interp(k_dct, k_model, P_model, left=0, right=0)
    w = windows.nuttall(2*M)[M:]
    pk_dct, cl = pfunc.pk_dct(fields_flag, dL, ml, M, r, w)

    kb_dct, mean_dct, err_dct, binned_th_dct, _ = pfunc.bin_power_spectrum(
        0, M, k_dct, pk_dct, P_cont_dct, NB
    )

    # -----------------------------
    # Save outputs
    # -----------------------------
    flag_name = flag_cfg["mode"]
    SCF = scf_cfg["SCF"]

    pfunc.save_data(
        f"{path}/fft",
        inp_signal=inp_signal,
        flag=flag_name,
        SCF=SCF,
        kb=kb_fft,
        mean=mean_fft,
        err=err_fft,
        binned_th=binned_th_fft,
    )

    pfunc.save_data(
        f"{path}/dct",
        inp_signal=inp_signal,
        flag=flag_name,
        SCF=SCF,
        kb=kb_dct,
        mean=mean_dct,
        err=err_dct,
        binned_th=binned_th_dct,
    )

    np.savez(
        f"{path}/CL_inp_signal-{inp_signal}_flag-{flag_name}_SCF-{SCF}.npz",
        cl=cl,
        inp_signal=inp_signal,
        flag=flag,
        SCF=SCF,
    )



In [7]:
# for inp_signal in INP_SIGNAL_CONFIGS:

#     print(f"\n=== Running inp_signal = {inp_signal} ===")

#     fields_tota = generate_fields(
#         inp_signal,
#         N=N,
#         Nrea=Nrea,
#         P_dft=P_dft,
#     )
#     # print(fields_tota[0][0])
    

#     for flag_key, flag_cfg in FLAG_CONFIGS.items():
#         for scf_key, scf_cfg in SCF_CONFIGS.items():

#             run_one_configuration(
#                 fields_tota=fields_tota,
#                 inp_signal=inp_signal,
#                 flag_cfg=flag_cfg,
#                 scf_cfg=scf_cfg,
#                 N=N, L=L, dL=dL, r=r, NB=NB,
#                 k_model=k_model,
#                 P_model=P_model,
#                 P_cont=P_cont,
#                 k_abs=k_abs,
#                 path="results",
#             )


In [8]:
# from tqdm.notebook import tqdm  # Use 'tqdm' if running in a script
from tqdm import tqdm

# Wrap your outer loop with tqdm
for inp_signal in tqdm(INP_SIGNAL_CONFIGS, desc="Signals"):

    print(f"\n=== Running inp_signal = {inp_signal} ===")
    
    fields_tota = generate_fields(
        inp_signal,
        N=N, Nrea=Nrea, P_dft=P_dft,
    )

    # You can also wrap inner loops if they are long
    for flag_key, flag_cfg in FLAG_CONFIGS.items():
        for scf_key, scf_cfg in SCF_CONFIGS.items():
            
            run_one_configuration(
                fields_tota=fields_tota,
                inp_signal=inp_signal,
                flag_cfg=flag_cfg,
                scf_cfg=scf_cfg,
                N=N, L=L, dL=dL, r=r, NB=NB,
                k_model=k_model,
                P_model=P_model,
                P_cont=P_cont,
                k_abs=k_abs,
                path="results1",
            )


Signals:   0%|                                                                                    | 0/3 [00:00<?, ?it/s]


=== Running inp_signal = 21cm ===
[FLAG=NOFLAG, SCF=None] flagged = 0.0%
[FLAG=NOFLAG, SCF=GP] flagged = 0.0%
Learned Ratios -> Smooth: 12747139.609, Fast: 0.113
Learned Ratios -> Smooth: 574127360.302, Fast: 0.132
Learned Ratios -> Smooth: 582.476, Fast: 0.248
Learned Ratios -> Smooth: 96.743, Fast: 0.341
Learned Ratios -> Smooth: 215224.926, Fast: 0.152
Learned Ratios -> Smooth: 41.592, Fast: 0.198
Learned Ratios -> Smooth: 95272832.448, Fast: 0.063
Learned Ratios -> Smooth: 2873.287, Fast: 0.156
Learned Ratios -> Smooth: 254168.863, Fast: 0.081
Learned Ratios -> Smooth: 1453.258, Fast: 0.444
Learned Ratios -> Smooth: 18296.202, Fast: 0.196
Learned Ratios -> Smooth: 158.526, Fast: 0.295
Learned Ratios -> Smooth: 1085798.440, Fast: 0.140
Learned Ratios -> Smooth: 94256.220, Fast: 0.132
Learned Ratios -> Smooth: 31535.271, Fast: 0.210
Learned Ratios -> Smooth: 603.878, Fast: 0.167


Signals:   0%|                                                                                    | 0/3 [00:15<?, ?it/s]


KeyboardInterrupt: 